# House predictions with linear regression

**Note:** This is the new version of the code, using scikit-learn instead of Turi Create (which was deprecated).
If you wish to see the original code in Turi Create that appears in the first edition of the book, please check [here](https://github.com/luisguiserrano/manning/blob/master/Chapter_03_Linear_Regression/DEPRECATED_House_price_predictions.ipynb).

### Importing packages

In [ ]:
# 线性回归与评估：对应 01.例子、02.损失函数（MSE/RMSE）、03.梯度下降
from sklearn.linear_model import LinearRegression  # 内部用最小二乘/梯度下降最小化平方损失
from sklearn.metrics import mean_squared_error     # MSE，见 02.损失函数.md
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

### Loading the dataset
For the next two cells, please run only one of them.
- Run the first cell if you cloned the Github Repo
- Run the second cell if you opened this as a Google Colab

In [ ]:
# 本地运行：仅当已克隆仓库并含有 Hyderabad.csv 时执行
file_path = 'Hyderabad.csv'
data = pd.read_csv(file_path)  # 房屋数据：Price（标签）、Area 等特征
data.head()

In [ ]:
# Colab 运行：从网络加载数据，二选一即可
url = "https://raw.githubusercontent.com/luisguiserrano/manning/master/Chapter_03_Linear_Regression/Hyderabad.csv"
data = pd.read_csv(url)
data.head()

In [ ]:
# 样本数 m、特征数 n，对应 01.例子「通用线性回归」符号约定
num_rows, num_cols = data.shape
print("The dataset has ", num_rows, "rows, and ", num_cols, " columns")

### Exploring the relationship between price and area

In [ ]:
# 单特征：面积 vs 价格，直观看「点到线的距离」（见 02.损失函数 图3.17/3.18/3.19）
plt.scatter(data['Area'], data['Price'])

In [ ]:
# 单特征线性回归：ŷ = w·x + b，对应 01.例子 斜率 m（每单位面积价格）、截距 b（基础价）
X = data[['Area']]   # 特征 x（面积）
y = data['Price']    # 标签 y（真实价格）

simple_model = LinearRegression()
# fit 内部最小化 MSE，用梯度下降或正规方程得到 w、b（见 02.损失函数、03.梯度下降）
simple_model.fit(X, y)

In [ ]:
# 学到的参数：intercept = b（y 轴截距），coef_ = w（斜率），即 01.例子 中的 b、m
print(f"y-intercept: {simple_model.intercept_}")
print(f"slope (coefficient of Area): {simple_model.coef_[0]}")

In [ ]:
# 画出拟合直线：对一系列 x 用 ŷ = w·x + b 预测，即 01.例子「最接近所有点」的那条线
area_range = np.linspace(X.min(), X.max(), 100).reshape(-1, 1)
predicted_prices = simple_model.predict(area_range)

plt.scatter(X, y, color='blue', label='Data Points')
plt.plot(area_range, predicted_prices, color='red', linewidth=2, label='Regression Line')

# Add labels and title
plt.xlabel('Area')
plt.ylabel('Price')
plt.title('Area vs. Price with Linear Regression')
plt.legend()

# Show the plot
plt.grid(True)
plt.show()

### Building a model that uses all the features

In [ ]:
# 多特征数据：对应 01.例子「通用线性回归」x⁽ⁱ⁾ = (x₁,...,xₙ)，n 个权重 + 1 个偏差
data

### Pre-processing the data
In order to train a model in scikit-learn, we need to pre-process the data:
1. Remove missing values
2. Scale and center the numerical features
3. One-hot encode the categorical features

In [ ]:
# 去掉缺失行（本数据集中缺失编码为 9），保证后续损失/梯度计算有意义
data_truncated = data[:2434]
data_truncated

In [ ]:
# 数值特征标准化（减均值除标准差），便于梯度下降收敛，见 03.梯度下降 学习率与尺度
data_scaled = data_truncated.copy()

area_mean = data_scaled['Area'].mean()
area_std = data_scaled['Area'].std()
data_scaled['Area'] = (data_scaled['Area'] - area_mean) / area_std

bedrooms_mean = data_scaled['No. of Bedrooms'].mean()
bedrooms_std = data_scaled['No. of Bedrooms'].std()
data_scaled['No. of Bedrooms'] = (data_scaled['No. of Bedrooms'] - bedrooms_mean) / bedrooms_std

print(data_scaled.head())

In [ ]:
# 类别特征 Location 做 one-hot，得到多列 0/1，通用模型 ŷ = w₁x₁+…+wₙxₙ+b 中每个 x 对应一列
data_scaled_encoded = pd.get_dummies(data_scaled, columns=['Location'], prefix='Location', dtype=int)

In [ ]:
# 标准化 + one-hot 后的特征矩阵，列数 = 数值特征 + Location 的各类别
print(data_scaled_encoded.head())

### Fit a Linear Regression model

In [ ]:
# 多特征线性回归：X_full 每行 x⁽ⁱ⁾，y_full 为标签；最小化 MSE 得到 n 个权重 + 1 个截距（01.例子 通用平方技巧）
X_full = data_scaled_encoded.drop('Price', axis=1)
y_full = data_scaled_encoded['Price']

model_predict_all = LinearRegression()
model_predict_all.fit(X_full, y_full)  # 内部梯度下降或正规方程，最小化 Σ(y−ŷ)²

In [ ]:
# 学到的 b（intercept）与各 w（coef_），即通用公式 ŷ = w₁x₁+…+wₙxₙ+b 中的参数
print("\nLinear Regression Model Coefficients (Predicting Price from all features):")
print(f"Intercept: {model_predict_all.intercept_}")
print("Coefficients for features:")
for feature, coef in zip(X_full.columns, model_predict_all.coef_):
    print(f"{feature}: {coef}")

In [ ]:
# 预测与评估：MSE = 均方误差，RMSE = 均方根误差（与价格同单位），见 02.损失函数「MAE/MSE/RMSE」「图3.23」
y_pred = model_predict_all.predict(X_full)
mse = mean_squared_error(y_full, y_pred)
rmse = np.sqrt(mse)
print(f"\nRoot Mean Squared Error (RMSE) of the model: {rmse}")

In [ ]:
# 查看第一行样本的特征向量（用于理解多特征输入）
X_full.loc[0]

In [ ]:
# 新样本预测：必须用与训练时相同的标准化与 one-hot，再代入 ŷ = w·x + b（01.例子 通用公式）
import pandas as pd

new_house_data = pd.DataFrame({'Area': [1000], 'No. of Bedrooms': [3]})
new_house_data['Area'] = (new_house_data['Area'] - area_mean) / area_std
new_house_data['No. of Bedrooms'] = (new_house_data['No. of Bedrooms'] - bedrooms_mean) / bedrooms_std

location_cols = [col for col in X_full.columns if col.startswith('Location_')]
new_house_location_dummies = pd.DataFrame(0, index=new_house_data.index, columns=location_cols)

if 'Location_Gachibowli' in location_cols:
    new_house_location_dummies['Location_Gachibowli'] = 1

new_house_processed = pd.concat([new_house_data, new_house_location_dummies], axis=1)
for col in X_full.columns:
    if col not in new_house_processed.columns:
        new_house_processed[col] = 0

new_house_processed = new_house_processed[X_full.columns]
predicted_price = model_predict_all.predict(new_house_processed)

print(f"\nPredicted price for a house with size 1000 and 3 bedrooms: {predicted_price[0]:,.2f}")